# Kubera Model Training (GPU)
Upload your `nifty50_1min_5years.parquet`, train the model, download `best_xgb_model.joblib` + `feature_cols.joblib`.

**Runtime → Change runtime type → T4 GPU**

In [ ]:
!pip install -q xgboost pandas_ta scikit-learn joblib seaborn pyarrow

In [ ]:
# Upload your parquet file
from google.colab import files
uploaded = files.upload()  # Select nifty50_1min_5years.parquet

In [ ]:
import pandas as pd
import pandas_ta as ta
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('All imports OK')
print(f'XGBoost version: {xgb.__version__}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════
DATA_FILE = 'nifty50_1min_5years.parquet'
TRAIN_RATIO = 0.8
HORIZON_BARS = 24

XGB_PARAMS = {
    'objective': 'binary:logistic',
    'max_depth': 6,
    'learning_rate': 0.05,
    'n_estimators': 300,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'tree_method': 'gpu_hist',   # GPU acceleration
    'device': 'cuda',
    'random_state': 42,
    'eval_metric': 'aucpr',
    'scale_pos_weight': 3.25,
}

print('Config set. GPU training enabled.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 1. LOAD & RESAMPLE
# ═══════════════════════════════════════════════════════════════
print(f'Loading {DATA_FILE}...')
df = pd.read_parquet(DATA_FILE)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)
print(f'Loaded: {df.shape}')

print('Resampling to 5-minute bars...')
df = df.set_index('date')
df = (
    df.groupby('symbol')
    .resample('5min')
    .agg({'open': 'first', 'high': 'max', 'low': 'min',
          'close': 'last', 'volume': 'sum'})
    .dropna()
    .reset_index()
)
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)

# Optimize memory
for col in df.select_dtypes(include=['float64']).columns:
    df[col] = df[col].astype('float32')
for col in df.select_dtypes(include=['int64']).columns:
    df[col] = df[col].astype('int32')

print(f'Resampled: {df.shape}')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 2. FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════════
def calc_symbol_features(df):
    df = df.copy()

    df['rsi'] = ta.rsi(df['close'], length=14)
    df['roc_5'] = ta.roc(df['close'], length=5)
    df['roc_20'] = ta.roc(df['close'], length=20)

    ema_20 = ta.ema(df['close'], length=20)
    ema_50 = ta.ema(df['close'], length=50)
    ema_100 = ta.ema(df['close'], length=100)
    df['ema_gap_20'] = (df['close'] - ema_20) / ema_20
    df['ema_gap_50'] = (df['close'] - ema_50) / ema_50
    df['ema_gap_100'] = (df['close'] - ema_100) / ema_100

    bbands = ta.bbands(df['close'], length=20, std=2)
    bbu = [c for c in bbands.columns if c.startswith('BBU_')][0]
    bbl = [c for c in bbands.columns if c.startswith('BBL_')][0]
    bbm = [c for c in bbands.columns if c.startswith('BBM_')][0]
    df['bb_width'] = (bbands[bbu] - bbands[bbl]) / bbands[bbm]
    df['bb_position'] = (df['close'] - bbands[bbl]) / (bbands[bbu] - bbands[bbl])

    df['atr'] = ta.atr(df['high'], df['low'], df['close'], length=14) / df['close']

    df['vol_sma_14'] = df['volume'] / ta.sma(df['volume'], length=14)
    df['mfi'] = ta.mfi(df['high'], df['low'], df['close'], df['volume'], length=14)
    df['vol_acceleration'] = df['volume'] / df['volume'].rolling(3).mean()

    df['return_1bar'] = df['close'].pct_change(1)
    df['return_3bar'] = df['close'].pct_change(3)
    df['return_6bar'] = df['close'].pct_change(6)
    df['return_12bar'] = df['close'].pct_change(12)
    df['return_24bar'] = df['close'].pct_change(24)
    df['volatility_6bar'] = df['return_1bar'].rolling(6).std()

    df['momentum_aligned'] = (
        (df['return_3bar'] > 0).astype(int) +
        (df['return_6bar'] > 0).astype(int) +
        (df['return_12bar'] > 0).astype(int)
    )

    adx = ta.adx(df['high'], df['low'], df['close'], length=14)
    df['adx'] = adx['ADX_14']
    df['dmp'] = adx['DMP_14']
    df['dmn'] = adx['DMN_14']
    df['di_diff'] = df['dmp'] - df['dmn']
    df['regime'] = (df['adx'] > 20).astype(int)

    macd = ta.macd(df['close'], fast=12, slow=26, signal=9)
    df['macd_hist'] = macd['MACDh_12_26_9']
    df['macd_slope'] = df['macd_hist'] - df['macd_hist'].shift(1)

    stochrsi = ta.stochrsi(df['close'], length=14)
    df['stochrsi_k'] = stochrsi['STOCHRSIk_14_14_3_3']
    df['stochrsi_d'] = stochrsi['STOCHRSId_14_14_3_3']

    vol_sum = df['volume'].groupby(df['date'].dt.date).cumsum()
    typical_price = (df['high'] + df['low'] + df['close']) / 3.0
    df['vwap'] = (typical_price * df['volume']).groupby(df['date'].dt.date).cumsum() / vol_sum
    df['vwap_dev'] = (df['close'] - df['vwap']) / df['vwap'] * 100

    df['day'] = df['date'].dt.date
    def _or_high(x, n=3):
        first_n = x.values[:min(n, len(x))]
        or_val = first_n.max() if len(first_n) > 0 else np.nan
        if len(x) <= n: return pd.Series(first_n, index=x.index)
        return pd.Series(list(first_n) + [or_val] * (len(x) - n), index=x.index)
    def _or_low(x, n=3):
        first_n = x.values[:min(n, len(x))]
        or_val = first_n.min() if len(first_n) > 0 else np.nan
        if len(x) <= n: return pd.Series(first_n, index=x.index)
        return pd.Series(list(first_n) + [or_val] * (len(x) - n), index=x.index)

    df['or_high'] = df.groupby('day')['high'].transform(_or_high)
    df['or_low'] = df.groupby('day')['low'].transform(_or_low)
    or_range = (df['or_high'] - df['or_low']).replace(0, np.nan)
    df['or_position'] = (df['close'] - df['or_low']) / or_range
    df['or_breakout'] = np.where(df['close'] > df['or_high'], 1,
                                  np.where(df['close'] < df['or_low'], -1, 0))

    df['hour'] = df['date'].dt.hour
    df['minute'] = df['date'].dt.minute
    df['minutes_from_open'] = ((df['hour'] - 9) * 60 + (df['minute'] - 15)).clip(lower=0)

    return df

print('Generating features...')
groups = []
symbols = df['symbol'].unique()
for i, (sym, grp) in enumerate(df.groupby('symbol')):
    if (i + 1) % 10 == 0:
        print(f'  [{i+1}/{len(symbols)}] {sym}')
    groups.append(calc_symbol_features(grp))

df = pd.concat(groups).reset_index(drop=True)
df = df.dropna()
print(f'Features done: {df.shape}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 2b. CROSS-SECTIONAL RANKS
# ═══════════════════════════════════════════════════════════════
print('Ranking features cross-sectionally...')
df['market_return'] = df.groupby('date')['return_1bar'].transform('mean')
df['relative_return'] = df['return_1bar'] - df['market_return']

rank_cols = [
    'rsi', 'roc_5', 'roc_20',
    'ema_gap_20', 'ema_gap_50', 'ema_gap_100',
    'bb_width', 'bb_position', 'atr',
    'vol_sma_14', 'mfi', 'vol_acceleration',
    'return_3bar', 'return_6bar', 'return_12bar', 'return_24bar',
    'adx', 'di_diff', 'macd_hist', 'macd_slope',
    'stochrsi_k', 'stochrsi_d', 'vwap_dev',
    'or_position', 'momentum_aligned'
]
for col in rank_cols:
    df[f'rank_{col}'] = df.groupby('date')[col].rank(pct=True)

print('Ranking done.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3. ATR-BASED LABELING (matches your main.py exactly)
# ═══════════════════════════════════════════════════════════════
ATR_PT = 3.0
ATR_SL = 1.5

print(f'ATR Labeling (PT: {ATR_PT}x ATR, SL: {ATR_SL}x ATR, Horizon: {HORIZON_BARS} bars)...')

df['label'] = 0
processed = []
for sym, group in df.groupby('symbol'):
    group = group.copy()
    close = group['close'].values
    atr_vals = group['atr'].values
    n = len(close)

    if n <= HORIZON_BARS:
        group['label'] = 0
        processed.append(group)
        continue

    labels = np.zeros(n)
    windows = np.lib.stride_tricks.sliding_window_view(close, HORIZON_BARS + 1)
    entry_prices = windows[:, 0:1]
    future_path = windows[:, 1:]
    pct_returns = (future_path - entry_prices) / entry_prices * 100.0

    bar_pt = atr_vals[:len(windows), np.newaxis] * 100.0 * ATR_PT
    bar_sl = atr_vals[:len(windows), np.newaxis] * 100.0 * ATR_SL
    group['pt_dynamic'] = atr_vals * 100.0 * ATR_PT
    group['sl_dynamic'] = atr_vals * 100.0 * ATR_SL

    tp_hits = pct_returns >= bar_pt
    sl_hits = pct_returns <= -bar_sl

    has_tp = np.any(tp_hits, axis=1)
    has_sl = np.any(sl_hits, axis=1)
    first_tp = np.where(has_tp, np.argmax(tp_hits, axis=1), HORIZON_BARS + 1)
    first_sl = np.where(has_sl, np.argmax(sl_hits, axis=1), HORIZON_BARS + 1)

    labels[:len(windows)] = np.where(
        first_tp < first_sl, 1,
        np.where(first_sl < first_tp, -1, 0)
    )
    group['label'] = labels
    processed.append(group)

df = pd.concat(processed).reset_index(drop=True)

dist = df['label'].value_counts(normalize=True)
print(f'Label distribution:\n{dist}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 4. FEATURE LIST
# ═══════════════════════════════════════════════════════════════
feature_cols = [
    'rsi', 'roc_5', 'roc_20',
    'ema_gap_20', 'ema_gap_50', 'ema_gap_100',
    'bb_width', 'bb_position',
    'atr', 'volatility_6bar',
    'vol_sma_14', 'mfi', 'vol_acceleration',
    'return_1bar', 'return_3bar', 'return_6bar',
    'return_12bar', 'return_24bar',
    'momentum_aligned',
    'adx', 'dmp', 'dmn', 'di_diff', 'regime',
    'macd_hist', 'macd_slope',
    'stochrsi_k', 'stochrsi_d',
    'vwap_dev',
    'or_position', 'or_breakout',
    'minutes_from_open',
    'relative_return',
    'rank_rsi', 'rank_roc_5', 'rank_roc_20',
    'rank_ema_gap_20', 'rank_ema_gap_50', 'rank_ema_gap_100',
    'rank_bb_width', 'rank_bb_position', 'rank_atr',
    'rank_vol_sma_14', 'rank_mfi', 'rank_vol_acceleration',
    'rank_return_3bar', 'rank_return_6bar',
    'rank_return_12bar', 'rank_return_24bar',
    'rank_adx', 'rank_di_diff',
    'rank_macd_hist', 'rank_macd_slope',
    'rank_stochrsi_k', 'rank_stochrsi_d',
    'rank_vwap_dev', 'rank_or_position',
    'rank_momentum_aligned'
]

# Verify all features exist
missing = [c for c in feature_cols if c not in df.columns]
if missing:
    print(f'ERROR: Missing features: {missing}')
else:
    print(f'All {len(feature_cols)} features present.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 5. TRAIN / VAL / CALIB / TEST SPLIT
# ═══════════════════════════════════════════════════════════════
unique_dates = sorted(df['date'].unique())
split_idx = int(len(unique_dates) * TRAIN_RATIO)
split_date = unique_dates[split_idx]

train_df = df[df['date'] < split_date].copy()
test_df = df[df['date'] >= split_date].copy()
print(f'Train: {len(train_df)} | Test: {len(test_df)}')

# Further split train
train_dates = sorted(train_df['date'].unique())
val_idx = int(len(train_dates) * 0.85)
calib_idx = int(len(train_dates) * 0.90)
val_date = train_dates[val_idx]
calib_date = train_dates[calib_idx]

tr = train_df[train_df['date'] < val_date]
val = train_df[(train_df['date'] >= val_date) & (train_df['date'] < calib_date)]
calib = train_df[train_df['date'] >= calib_date]

X_train = tr[feature_cols]
y_train = (tr['label'] == 1).astype(int)
X_val = val[feature_cols]
y_val = (val['label'] == 1).astype(int)
X_calib = calib[feature_cols]
y_calib = (calib['label'] == 1).astype(int)
X_test = test_df[feature_cols]
y_test = (test_df['label'] == 1).astype(int)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Calib: {len(X_calib)} | Test: {len(X_test)}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 6. TRAIN XGBoost (GPU)
# ═══════════════════════════════════════════════════════════════
print('Training XGBoost on GPU...')

params = XGB_PARAMS.copy()

# Check if GPU is available, fallback to CPU
try:
    test_model = xgb.XGBClassifier(tree_method='gpu_hist', device='cuda', n_estimators=1)
    test_model.fit(X_train.head(100), y_train.head(100))
    print('GPU detected. Training with gpu_hist.')
except Exception:
    print('No GPU found. Falling back to CPU (hist).')
    params['tree_method'] = 'hist'
    params.pop('device', None)

base_model = xgb.XGBClassifier(**params)
base_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=True
)

print('\nBase model trained.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7. CALIBRATE (sigmoid, cv=3 — matches your model_trainer.py)
# ═══════════════════════════════════════════════════════════════
print(f'Calibrating on {len(X_calib)} rows (sigmoid, cv=3)...')

calibrated = CalibratedClassifierCV(base_model, method='sigmoid', cv=3)
calibrated.fit(X_calib, y_calib)

print('Calibration done.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8. EVALUATE ON TEST SET
# ═══════════════════════════════════════════════════════════════
y_pred_proba = calibrated.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba > 0.5).astype(int)

print(classification_report(y_test, y_pred))
print(f'ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}')

# Probability distribution
print(f'\n--- Probability Distribution (Test Set) ---')
for q in [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]:
    print(f'  {int(q*100)}th: {np.quantile(y_pred_proba, q):.4f}')
print(f'  Max:  {y_pred_proba.max():.4f}')
print(f'  > 0.10: {(y_pred_proba > 0.10).sum():,}')
print(f'  > 0.20: {(y_pred_proba > 0.20).sum():,}')
print(f'  > 0.30: {(y_pred_proba > 0.30).sum():,}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 9. FEATURE IMPORTANCE
# ═══════════════════════════════════════════════════════════════
try:
    imp = calibrated.calibrated_classifiers_[0].estimator.feature_importances_
    feat_imp = pd.Series(imp, index=feature_cols).sort_values(ascending=False)
    plt.figure(figsize=(10, 8))
    sns.barplot(x=feat_imp.values[:15], y=feat_imp.index[:15])
    plt.title('Top 15 Feature Importances')
    plt.tight_layout()
    plt.savefig('feature_importance.png')
    plt.show()
    print('Top 5:', feat_imp.head().to_dict())
except Exception as e:
    print(f'Could not get feature importance: {e}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 10. SAVE & DOWNLOAD
# ═══════════════════════════════════════════════════════════════
joblib.dump(calibrated, 'best_xgb_model.joblib')
joblib.dump(feature_cols, 'feature_cols.joblib')

print('Saved: best_xgb_model.joblib, feature_cols.joblib')
print('\nDownloading to your machine...')

from google.colab import files
files.download('best_xgb_model.joblib')
files.download('feature_cols.joblib')
files.download('feature_importance.png')

print('\n✅ Done! Place both .joblib files in your ml_modl directory.')
print('Then run: python kite_bridge.py')